In [48]:
import random
import math
from typing import Tuple, Optional

In [49]:
def gcd(a: int, b: int) -> int:
    """Greatest common divisor."""
    a, b = abs(a), abs(b)
    while b:
        a, b = b, a % b
    return a


def mod_pow(a: int, e: int, n: int) -> int:
    """Fast modular exponentiation: a^e mod n."""
    return pow(a, e, n)


In [63]:
import random

def is_probable_prime_fermat(n: int, rounds: int = 10) -> bool:
    if n < 2:
        return False
    if n in (2, 3):
        return True
    if n % 2 == 0:
        return False

    for _ in range(rounds):
        a = random.randrange(2, n - 1)
        if gcd(a, n) != 1:
            return False  # composite
        if pow(a, n - 1, n) != 1:
            return False  # composite
    return True  # probably prime


In [65]:
is_probable_prime_fermat(12)

False

In [66]:
def jacobi(a: int, n: int) -> int:
    """
    Compute Jacobi symbol (a/n) for odd n >= 3.
    Returns: -1, 0, or 1.

      - factor out powers of 2 from a and adjust sign by n mod 8
      - apply quadratic reciprocity when swapping (a, n)
      - reduce a mod n
    """
    if n <= 0 or n % 2 == 0:
        raise ValueError("Jacobi symbol is defined here for positive odd n.")

    a %= n
    result = 1

    while a != 0:
        # Step A: remove factors of 2 from a
        while a % 2 == 0:
            a //= 2
            r = n % 8
            if r == 3 or r == 5:
                result = -result

        # Step B: quadratic reciprocity swap
        a, n = n, a  # swap

        if (a % 4 == 3) and (n % 4 == 3):
            result = -result

        a %= n

    # if n == 1 => result, else a became 0 with n>1 => gcd != 1 => 0
    return result if n == 1 else 0

In [67]:
print(jacobi(1001, 9907))

-1


In [68]:
def is_probable_prime_solovay_strassen(n: int, rounds: int = 10, rng: Optional[random.Random] = None) -> bool:
    """
    Solovay–Strassen:
      pick random a in [2, n-2]
      if gcd(a, n) > 1 -> composite
      r = a^((n-1)/2) mod n
      s = jacobi(a, n)
      compare r with s (mod n)
    """
    if rng is None:
        rng = random.Random()

    if n < 2:
        return False
    if n in (2, 3):
        return True
    if n % 2 == 0:
        return False

    for _ in range(rounds):
        a = rng.randrange(2, n - 1)
        g = gcd(a, n)
        if g > 1:
            return False

        r = mod_pow(a, (n - 1) // 2, n)
        s = jacobi(a, n)
        if s == 0:
            return False

        # s is -1 or 1; compare modulo n
        s_mod = s % n
        if r != s_mod:
            return False

    return True

In [69]:
print(is_probable_prime_solovay_strassen(11,10))

True


In [54]:
def is_probable_prime_miller_rabin(n: int, rounds: int = 10, rng: Optional[random.Random] = None) -> bool:
    """
    Miller–Rabin:
      write n-1 = 2^s * d with d odd
      pick random a in [2, n-2]
      x = a^d mod n
      if x in {1, n-1} pass round
      else repeatedly square x s-1 times:
        if x becomes n-1 -> pass round
        if x becomes 1 early -> composite
      otherwise composite
    """
    if rng is None:
        rng = random.Random()

    if n < 2:
        return False
    if n in (2, 3):
        return True
    if n % 2 == 0:
        return False

    # n-1 = 2^s * d
    d = n - 1
    s = 0
    while d % 2 == 0:
        d //= 2
        s += 1

    for _ in range(rounds):
        a = rng.randrange(2, n - 1)
        x = mod_pow(a, d, n)

        if x == 1 or x == n - 1:
            continue

        composite = True
        for _ in range(s - 1):
            x = (x * x) % n
            if x == n - 1:
                composite = False
                break
            if x == 1:
                return False  # non-trivial square root of 1 => composite

        if composite:
            return False

    return True